In [24]:
# Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [25]:
steam = pd.read_csv("../data/processed/steam_cleaned.csv")
steam.head()

,appid,name,release_date,english,developer,publisher,platforms,required_age,categories,genres,...,average_playtime,median_playtime,owners,price,total_reviews,review_score,success,log_success,log_reviews,log_playtime
0,10,Counter-Strike,2000-11-01,1,Valve,Valve,windows;mac;linux,0,Multi-player;Online Multi-Player;Local Multi-P...,Action,...,17612,317,10000000-20000000,7.19,127873,0.973888,124534.0,11.732342,11.758801,9.776393
1,20,Team Fortress Classic,1999-04-01,1,Valve,Valve,windows;mac;linux,0,Multi-player;Online Multi-Player;Local Multi-P...,Action,...,277,62,5000000-10000000,3.99,3951,0.839787,3318.0,8.107419,8.281977,5.627621
2,30,Day of Defeat,2003-05-01,1,Valve,Valve,windows;mac;linux,0,Multi-player;Valve Anti-Cheat enabled,Action,...,187,34,5000000-10000000,3.99,3814,0.895648,3416.0,8.136518,8.246696,5.236442
3,40,Deathmatch Classic,2001-06-01,1,Valve,Valve,windows;mac;linux,0,Multi-player;Online Multi-Player;Local Multi-P...,Action,...,258,184,5000000-10000000,3.99,1540,0.826623,1273.0,7.149917,7.340187,5.556828
4,50,Half-Life: Opposing Force,1999-11-01,1,Gearbox Software,Valve,windows;mac;linux,0,Single-player;Multi-player;Valve Anti-Cheat en...,Action,...,624,415,5000000-10000000,3.99,5538,0.947996,5250.0,8.566174,8.619569,6.437752


# Retention Pattern Analysis

Questions to answer in this notebook:

* What do top-retention games have in common?
* What do low-retention games have in common?
* Which genres appear most often?
* Are they more likely to be multiplayer, action, RPG, sandbox, strategy?
* Do they also have higher popularity and playtime?

In [26]:
f2p = steam[steam["price"] == 0].copy()
paid = steam[steam["price"] > 0].copy()

In [27]:
f2p.to_csv("f2p_games.csv", index=False)
paid.to_csv("paid_games.csv", index=False)

## Free to play games

In [28]:
f2p = pd.read_csv("../data/processed/f2p_games.csv")

f2p.head()

,appid,name,release_date,english,developer,publisher,platforms,required_age,categories,genres,...,average_playtime,median_playtime,owners,price,total_reviews,review_score,success,log_success,log_reviews,log_playtime
0,280,Half-Life: Source,2004-06-01,1,Valve,Valve,windows;mac;linux,0,Single-player,Action,...,190,214,2000000-5000000,0.0,4820,0.781535,3767.0,8.234300,8.480737,5.252273
1,340,Half-Life 2: Lost Coast,2005-10-27,1,Valve,Valve,windows;mac;linux,0,Single-player;Commentary available,Action,...,46,29,10000000-20000000,0.0,6803,0.850066,5783.0,8.662851,8.825266,3.850148
2,360,Half-Life Deathmatch: Source,2006-05-01,1,Valve,Valve,windows;mac;linux,0,Multi-player;Valve Anti-Cheat enabled,Action,...,102,81,5000000-10000000,0.0,1835,0.742234,1362.0,7.217443,7.515345,4.634729
3,440,Team Fortress 2,2007-10-10,1,Valve,Valve,windows;mac;linux,0,Multi-player;Cross-Platform Multiplayer;Steam ...,Action;Free to Play,...,8495,623,20000000-50000000,0.0,549915,0.938107,515879.0,13.153629,13.217521,9.047351
4,570,Dota 2,2013-07-09,1,Valve,Valve,windows;mac;linux,0,Multi-player;Co-op;Steam Trading Cards;Steam W...,Action;Free to Play;Strategy,...,23944,801,100000000-200000000,0.0,1005586,0.858710,863507.0,13.668758,13.821082,10.083515


In [29]:
f2p_retention_threshold_high = f2p["average_playtime"].quantile(0.9)
f2p_retention_threshold_low = f2p["average_playtime"].quantile(0.2)

f2p_high_retention = f2p[f2p["average_playtime"] >= f2p_retention_threshold_high].copy()
f2p_low_retention = f2p[f2p["average_playtime"] <= f2p_retention_threshold_low].copy()

In [30]:
f2p_retention_threshold_low


np.float64(0.0)

In [31]:
f2p_low_retention["genres"].mean

<bound method Series.mean of 9                                    Casual;Indie
13                                       Strategy
14                                   Casual;Indie
16                                      Adventure
17                                      Adventure
                          ...                    
2555                   Casual;Free to Play;Sports
2556                    Action;Free to Play;Indie
2557          Action;Adventure;Free to Play;Indie
2558    Adventure;Free to Play;Indie;RPG;Strategy
2559                                 Free to Play
Name: genres, Length: 1738, dtype: object>

In [32]:
f2p_retention_comparison = pd.DataFrame({
    "high_retention_f2p": f2p_high_retention[["total_reviews", "review_score"]].mean(),
    "low_retention_f2p": f2p_low_retention[["total_reviews", "review_score"]].mean()
})

f2p_retention_comparison.index = ["popularity", "satisfaction"]
f2p_retention_comparison

,high_retention_f2p,low_retention_f2p
popularity,32115.144531,109.205984
satisfaction,0.746057,0.717724


* ~30k reviews on high retention games vs ~109 reviews on low retention games
* Only a 3% difference on customer satisfaction from high to low retention games.

Free-to-play games with high retention tend to achieve significantly higher popularity, while differences in satisfaction are relatively small. This suggests that engagement and player base size are the primary drivers of retention, rather than small variations in perceived quality.

In [33]:
# Genres retention

f2p_high_retention["genres"].value_counts().head(10)

genres
Free to Play;Massively Multiplayer;RPG                     12
Action;Free to Play;Indie                                  12
Action;Free to Play                                        10
Action;Adventure;Free to Play;Massively Multiplayer;RPG     7
Adventure;Free to Play;Massively Multiplayer;RPG            6
Action;Free to Play;Massively Multiplayer                   6
Casual;Free to Play;Indie                                   6
Free to Play;Strategy                                       5
Free to Play                                                5
Strategy                                                    5
Name: count, dtype: int64

In [34]:
f2p_low_retention["genres"].value_counts().head(10)

genres
Action;Free to Play;Indie              64
Casual;Free to Play;Indie              57
Indie                                  45
Action;Indie                           42
Casual;Indie                           40
Adventure;Free to Play;Indie           40
Adventure                              35
Action;Casual;Free to Play;Indie       34
Adventure;Indie                        33
Adventure;Casual;Free to Play;Indie    31
Name: count, dtype: int64

Since genres like Action, casual and Indie appear in both groups I'll split the genres to make sure this is accurate ( here they're in combinations instead of individual components)

In [35]:
# splitting genres using explode

f2p_high_genres = f2p_high_retention.assign(
    genre=f2p_high_retention["genres"].str.split(";")
).explode("genre")

f2p_low_genres = f2p_low_retention.assign(
    genre= f2p_low_retention["genres"].str.split(";")
).explode("genre")

In [36]:
f2p_high_pct = f2p_high_genres["genre"].value_counts(normalize=True)
f2p_high_pct.head(10)

genre
Free to Play             0.223897
Action                   0.134553
Indie                    0.114101
Massively Multiplayer    0.104413
RPG                      0.096878
Adventure                0.080732
Strategy                 0.079656
Casual                   0.065662
Simulation               0.043057
Early Access             0.026911
Name: proportion, dtype: float64

In [37]:
f2p_low_pct = f2p_low_genres["genre"].value_counts(normalize=True)
f2p_low_pct.head(10)

genre
Indie                    0.198843
Free to Play             0.180947
Casual                   0.117317
Action                   0.106652
Adventure                0.091829
Simulation               0.060918
Strategy                 0.056038
RPG                      0.050615
Early Access             0.048265
Massively Multiplayer    0.027838
Name: proportion, dtype: float64

In [38]:
genre_comparison = pd.DataFrame({
    "high_retention_pct": f2p_high_pct,
    "low_retention_pct": f2p_low_pct
}).fillna(0)

genre_comparison["difference"] = (
    genre_comparison["high_retention_pct"] -
    genre_comparison["low_retention_pct"]
)

genre_comparison.sort_values("difference", ascending=False).head(10)

,high_retention_pct,low_retention_pct,difference
genre,,,
Massively Multiplayer,0.104413,0.027838,0.076575
RPG,0.096878,0.050615,0.046264
Free to Play,0.223897,0.180947,0.042949
Action,0.134553,0.106652,0.027901
Strategy,0.079656,0.056038,0.023618
Nudity,0.002153,0.001988,0.000164
Photo Editing,0.000000,0.000542,-0.000542
Game Development,0.000000,0.000542,-0.000542
Audio Production,0.000000,0.000542,-0.000542


High-retention free-to-play games are strongly associated with genres such as Massively Multiplayer, RPG, and Strategy. These genres share a common characteristic: they are systems-driven and designed for long-term engagement. In contrast, genres without strong progression or social mechanics are less likely to retain players over time.

Action alone has a small difference but still positive, if joined with other high retention genres like RPG, multiplayer or strategy systems.

## Paid games

In [39]:
paid_high_threshold = paid["average_playtime"].quantile(0.9)
paid_low_threshold = paid["average_playtime"].quantile(0.1)

paid_high_retention = paid[paid["average_playtime"] >= paid_high_threshold].copy()
paid_low_retention = paid[paid["average_playtime"] < paid_high_threshold].copy()

print(len(paid_high_retention), len(paid_low_retention))

2457 22058


In [40]:
paid_high_genres = paid_high_retention.assign(
    genre=paid_high_retention["genres"].str.split(";")
).explode("genre")

paid_low_genres = paid_low_retention.assign(
    genre=paid_low_retention["genres"].str.split(";")
).explode("genre")

In [41]:
paid_high_pct = paid_high_genres["genre"].value_counts(normalize=True)

paid_low_pct = paid_low_genres["genre"].value_counts(normalize=True)

In [42]:
# Comparison

paid_genre_comparison = pd.DataFrame({
    "high_retention_pct": paid_high_pct,
    "low_retention_pct": paid_low_pct
}).fillna(0)

paid_genre_comparison["difference"] = (
    paid_genre_comparison["high_retention_pct"] -
    paid_genre_comparison["low_retention_pct"]
)

paid_genre_comparison.sort_values("difference", ascending=False).head(10)

,high_retention_pct,low_retention_pct,difference
genre,,,
RPG,0.087839,0.053107,0.034732
Action,0.190860,0.157859,0.033001
Strategy,0.085825,0.067879,0.017946
Adventure,0.146708,0.135742,0.010966
Massively Multiplayer,0.009760,0.004631,0.005128
Nudity,0.006662,0.003331,0.003330
Simulation,0.072037,0.068968,0.003069
Sexual Content,0.005112,0.003071,0.002041
Photo Editing,0.000465,0.000098,0.000367


In [43]:
genres_exploded = steam.assign(
    Genres=steam['genres'].str.split(';')
).explode('Genres')

In [44]:
genre_stats = genres_exploded.groupby("Genres").agg({
    "average_playtime": "mean",
    "success": "mean",
    "appid": "count"
}).reset_index()

genre_stats.rename(columns={
    "appid": "game_count"
}, inplace=True)

genre_stats = genre_stats[genre_stats["game_count"] > 20]

genre_stats.to_csv("../data/processed/genre_stats.csv", index=False)

## Business Recommendations

The analysis reveals that retention drivers differ significantly between free-to-play and paid games. Free-to-play games are strongly driven by multiplayer and large-scale engagement systems, while paid games rely more on gameplay depth and replayability. Although genres such as RPG and Strategy contribute to retention in both models, their impact is more pronounced in free-to-play games, where engagement must be sustained over longer periods.

Based on this findings we can strongly recommend Steam and Developers/Publishers to follow 3 strategies:

    Free-to-play strategy: Prioritize Multiplayer & Social features
        This is based on: 
                    Strong MMO signal (+7.6%);
                    High retention in Multiplayer games;
                    E.g of games like this: CS.GO, Dota2, Warframe.

    Paid games strategy: Promote replayability & strategic depth
        This is based on: 
                    Strategy and RPG retention in games;
                    E.g of games like this: Terraria, GTA V, Factorio.

    All games strategy: Encourage progression and RPG systems
        This is based on: 
                    RPG is a strong driver in both f2p and paid games;
                    High retention linked to progress systems;
                    (Bigger retention found in games that have bigger progression genres and rpg systems where players can role play in a fictional setting and games that also focus on narrative)

    We also discover that they should not prioritize low-engagement genres because, games lacking depth or long-term engagement systems, particularly those focused on short or casual experiences, should be approached cautiously, as they are less likely to retain players.
        Based on:
            Casual & Indie overrepresented in low-retention; 
            Groups with weak engagement patterns;
            E.g of games like this: Gumboy, Earthshakers. 